# VirtualTissue v2: annotated object → context → guidebook → simulation

Run cells in order. The default uses 24 **synthetic** cells and no API key. Replace the input with your reviewed annotated region after reading `docs/v2/WORKFLOW.md`. Uncalibrated action preferences and simulated states do not establish treatment efficacy. Expression stays fixed at t0.


In [ ]:
from pathlib import Path
import subprocess, sys
# Local Jupyter: launch inside the repository. Colab: this clones the v2 branch.
REPO = Path.cwd()
if not (REPO / 'src/virtualtissue_xenium').is_dir():
    REPO = REPO.parent if (REPO.parent / 'src/virtualtissue_xenium').is_dir() else Path('/content/VirtualTissue-Xenium')
if not (REPO / 'src/virtualtissue_xenium').is_dir():
    subprocess.run(['git', 'clone', '--branch', 'v2/context-guidebook', 'https://github.com/hylixinsights/VirtualTissue-Xenium.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
from virtualtissue_xenium.v2.inputs import load_cells, prepare, read_json, write_json
from virtualtissue_xenium.v2.simulation import preflight, run
print('Repository:', REPO)


## 1. Select the annotated input and context

For `.h5ad`, install the scientific adapters first with `pip install -e '.[xenium]'` (from the repository). Set the count layer and annotation names below. For Xenium raw output, use the reviewed annotation CSV workflow in `docs/v2/DATASETS.md`. For single-nucleus data without measured positions, explicitly choose `assay=snrna` and `interaction_mode=well_mixed`. Never infer a spatial layout from a transcriptomic embedding.


In [ ]:
INPUT = REPO / 'examples/v2/synthetic_cells.json'
CONTEXT = REPO / 'examples/v2/context_melanoma_control.json'
OUTPUT = REPO / 'outputs/v2/notebook_demo'
# For AnnData, adapt these keys and use layer='counts' if X is not raw counts.
cells, genes = load_cells(INPUT, cell_type_key='cell_type', lineage_key='lineage', sample_key=None, layer=None, max_cells=1000)
context = read_json(CONTEXT)
bundle = prepare(cells, context, assayed_genes=genes)
write_json(OUTPUT / 'prepared.json', bundle)
preflight(bundle)


## 2. Review the guidebook before simulating

Inspect coverage, cell-type mappings and action assumptions. Absent panel evidence remains missing. The expression-to-weight rule and contextual modifiers are assumptions, not transition probabilities learned from observed outcomes. Rebuild after changing the context or guidebook parameters.


In [ ]:
bundle['guidebook']['by_cell_type']


In [ ]:
# Inspect one cell and the exact rules producing its weights.
first_id = next(iter(bundle['guidebook']['cells']))
bundle['guidebook']['cells'][first_id]


## 3. Run a free local preview

Every cell sees the same snapshot. The next state is committed only after all proposals are ready. Changing the timestep changes the assumed model; the hour labels are not calibrated kinetics.


In [ ]:
local_folder = OUTPUT / 'local'
report = run(bundle, local_folder, steps=12, seed=21, resume=(local_folder/'config.json').exists())
report


In [ ]:
from IPython.display import IFrame, display
# In local Jupyter use the notebook server's file link; in Colab display HTML inline.
from IPython.display import HTML
replay = local_folder / 'replay.html'
display(HTML(replay.read_text()))
print('Standalone replay:', replay)


## 4. Optional bounded live Jev run

Set `RUN_JEV=True` only after reviewing the input and first-round estimate. The API receives tissue/disease/perturbation metadata and summarized per-cell evidence. The key remains in memory. A failed or unresolved request is never automatically resent; successful cached answers can be reused with identical settings.


In [ ]:
RUN_JEV = False
if RUN_JEV:
    import getpass, os
    from virtualtissue_xenium.v2.provider import JevProvider
    key = os.environ.get('TYPESAFE_API_KEY') or getpass.getpass('Jev API key: ')
    jev_folder = OUTPUT / 'jev'
    provider = JevProvider(key, jev_folder/'jev_journal', {'bundle':bundle['fingerprint'],'steps':2,'seed':21}, max_attempts=10, workers=2)
    key = ''
    jev_report = run(bundle, jev_folder, steps=2, seed=21, mode='jev', provider=provider, resume=(jev_folder/'config.json').exists())
    provider.key = ''
    display(jev_report)
else:
    print('No Jev requests sent. Local preview is complete.')


## 5. Next: matched experiments and validation

Create a second context using the same measured cells. Change only the explicit intervention/assumption being tested, retain the same graph/timestep/scoring/sampling and seeds, then run both arms. Compare their latest common endpoint with `vtx2 compare` or `virtualtissue_xenium.v2.cli.compare`. Repeat across seeds and include prompt-only, guidebook-only, no-modifier and recognition-off controls.

A higher action probability caused by an imposed weight modifier is not independent biological evidence. Review annotations and validate on independent perturbation outcomes before predictive interpretation. The task dependencies, parallel workstreams and remaining calibration work are in `docs/v2/WORKFLOW.md`.
